# 2_models/01b — Somatic feature comparison

Runs the `somatic` modality of `run_feature_comp_task.py` across the feature-comparison manifest,
as a notebook rather than through `slurm/launch_feature_comp.sh`.

`somatic` is the one modality that sits awkwardly between the two SLURM classes. Its penalized
block is a gene-by-alteration panel (`*_AMP`, `*_DEL`, `*_SNV`, `*_SV`, `*_FUSION`) whose width is
data-dependent, so unlike `stage`/`treatment`/`metburden` it is normally well over the 50-column
threshold at which `run_feature_comp_task.py` stops forcing `n_jobs=1`. That means **it is not
single-core work** and the sizing here differs from `01_feature_comparison.ipynb`: fewer workers,
each given several cores, rather than many single-core workers.

The width cell below reports the actual panel size, so you can confirm which side of the threshold
this data lands on before committing to a sizing.

**Safe to run alongside the arrays.** `run_feature_comp_task.py` skips any scheme/event/modality
whose four output files already exist, so with `OVERWRITE = False` this and a running
`MODALITY_CLASS=big` array step around each other's finished work. A triple begun simultaneously on
both sides is computed twice — deterministic outputs, so that costs CPU, not correctness.

Run this in a compute-backed Jupyter session — **not** on an ERISTwo login node.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config  # noqa: E402


def check_inputs(preconditions: list[tuple[str, str]]) -> list[str]:
    """Report presence of each (label, path). Returns the missing labels; never raises."""
    missing = []
    for label, path in preconditions:
        ok = os.path.exists(path)
        if not ok:
            missing.append(label)
        print(f"[{'ok ' if ok else 'MISSING'}] {label:<22} {path}")
    print(f"\n{'All inputs present.' if not missing else str(len(missing)) + ' missing: ' + ', '.join(missing)}")
    return missing


def run_module(module: str, args: list[str] | None = None, env: dict | None = None,
               capture: bool = False) -> dict:
    """Run `python -m module` from REPO_ROOT. Returns {returncode, wall_s, stdout}."""
    cmd = [sys.executable, "-m", module, *(args or [])]
    started = time.perf_counter()
    run_env = {**os.environ, "PYTHONUNBUFFERED": "1", **(env or {})}
    kwargs = dict(cwd=str(REPO_ROOT), env=run_env)
    if capture:
        kwargs.update(text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    proc = subprocess.run(cmd, **kwargs)
    return {"returncode": proc.returncode, "wall_s": time.perf_counter() - started,
            "stdout": (proc.stdout or "") if capture else ""}


print(f"repo root: {REPO_ROOT}")
print(f"Python:  {sys.executable}")

## Configuration

`MAX_ITER` and `BACKEND` mirror the launcher's `COXNET_MAX_ITER` / `COXNET_BACKEND` defaults; the
alpha/l1 grid is left at the script's defaults so results stay comparable with the array's.

`N_JOBS` is the CV-fold parallelism *within* one task, and `MAX_WORKERS` the number of tasks in
flight. They multiply: the pool consumes roughly `N_JOBS x MAX_WORKERS` cores. The defaults below
assume a modest interactive allocation — raise them to match what you actually reserved, and do not
size against `os.cpu_count()` on a shared node, which reports the whole machine.

In [ ]:
import schemes
from anchors import anchor_suffix
from pipelines.training.slurm_array_utils import _feature_file

MODULE = "pipelines.training.run_feature_comp_task"
MODALITY = "somatic"

ANCHOR = "treatment"
MAX_ITER = 1000
BACKEND = "threading"
OVERWRITE = False   # False leaves the arrays' finished work alone
VERBOSE = False     # True replays each subprocess's captured output when it finishes

# Unlike the small three, somatic is genuinely multi-core (see the width cell below).
# These multiply: the pool wants about N_JOBS * MAX_WORKERS cores.
N_JOBS = 4
MAX_WORKERS = 2

# Trial-run controls. Both None for the full manifest.
MAX_TASKS = None            # e.g. 4 for a smoke run
SCHEMES_FILTER = None       # e.g. {"death_met"}

MANIFEST = REPO_ROOT / "slurm" / "slurm_manifests" / f"feature_comp_tasks{anchor_suffix(ANCHOR)}.tsv"

print(f"anchor:     {ANCHOR}")
print(f"modality:   {MODALITY}")
print(f"manifest:   {MANIFEST}")
print(f"overwrite:  {OVERWRITE}   n_jobs: {N_JOBS} x workers: {MAX_WORKERS} "
      f"= up to {N_JOBS * MAX_WORKERS} cores")

## Preconditions

The manifest plus every feature file this modality's cohort restriction touches. Note that the list
is wider than the files `somatic` itself reads: `load_feature_modalities_df` restricts to
`_get_common_feature_mrns`, which intersects MRNs across **somatic, PRS, stage and treatment**. A
missing or unreadable file among those breaks a `somatic` run even though the model never uses its
columns. This cell does not raise.

In [ ]:
missing = check_inputs([
    ("manifest",          str(MANIFEST)),
    ("somatic",           _feature_file("complete_somatic_data_df.csv.gz", ANCHOR)),
    ("germline / PRS",    os.path.join(config.FEATURE_PATH, "complete_germline_data_df.csv.gz")),
    ("cancer stage",      _feature_file("cancer_stage_df.csv.gz", ANCHOR)),
    ("treatment by line", _feature_file("categorical_treatment_data_by_line.csv.gz", ANCHOR)),
    ("cancer types",      _feature_file("cancer_type_df.csv.gz", ANCHOR)),
])

if missing:
    print("\nThe cohort intersection reads all of the above, so any missing file fails every\n"
          "task identically — a uniform sweep of failures, not a scattered few.")

## Panel width

How many penalized columns `somatic` actually has, and therefore whether
`run_feature_comp_task.py` will honour `N_JOBS` or override it to 1.

The threshold is 50: `n_jobs = 1 if len(cfg["penalized_cols"]) < 50 else _get_n_jobs(args.n_jobs)`.
If the count comes back under 50, set `N_JOBS = 1` and raise `MAX_WORKERS` instead — the extra
cores would otherwise sit idle.

In [ ]:
import polars as pl

SOMATIC_SUFFIXES = ("_AMP", "_DEL", "_SNV", "_SV", "_FUSION")

somatic_path = _feature_file("complete_somatic_data_df.csv.gz", ANCHOR)
somatic_columns = pl.read_csv(somatic_path, n_rows=0).columns
somatic_cols = [c for c in somatic_columns if c.endswith(SOMATIC_SUFFIXES)]

by_suffix = {sfx: sum(1 for c in somatic_cols if c.endswith(sfx)) for sfx in SOMATIC_SUFFIXES}
print(f"{len(somatic_cols)} penalized somatic columns")
for suffix, n in by_suffix.items():
    print(f"  {suffix:<8} {n:>5}")

if len(somatic_cols) < 50:
    print(f"\n[note] under the 50-column threshold — run_feature_comp_task will force n_jobs=1\n"
          f"       regardless of N_JOBS={N_JOBS}. Set N_JOBS=1 and raise MAX_WORKERS.")
else:
    print(f"\n[ok] over the 50-column threshold — N_JOBS={N_JOBS} will be honoured.")

## Build the task list

The manifest is `scheme<TAB>event`, with an optional third field pinning a row to one modality. A
pinned row is honoured here only when it names `somatic`; rows pinned to another modality are
dropped.

In [ ]:
SCHEME_ALIASES = {"icd3": "icd3_post", "icd4": "icd4_post", "phecode": "phecode_post"}

tasks: list[tuple[str, str]] = []
rows = skipped_pinned = 0

for line_number, raw in enumerate(MANIFEST.read_text().splitlines(), 1):
    if not raw.strip():
        continue
    fields = raw.split("\t")
    if len(fields) not in (2, 3) or not all(fields[:2]):
        raise ValueError(f"{MANIFEST}:{line_number}: expected scheme<TAB>event[<TAB>modality]")
    scheme = SCHEME_ALIASES.get(fields[0], fields[0])
    event = fields[1]
    pinned = fields[2] if len(fields) == 3 and fields[2] else None
    rows += 1

    if pinned is not None and pinned != MODALITY:
        skipped_pinned += 1
        continue
    if SCHEMES_FILTER and scheme not in SCHEMES_FILTER:
        continue
    tasks.append((scheme, event))

if MAX_TASKS is not None:
    tasks = tasks[:MAX_TASKS]

print(f"{rows} manifest row(s) -> {len(tasks)} {MODALITY} task(s)")
if skipped_pinned:
    print(f"  ({skipped_pinned} row(s) pinned to another modality, dropped)")
if MAX_TASKS is not None:
    print(f"  (truncated to MAX_TASKS={MAX_TASKS})")

## Pre-flight filtering

Two gates, applied **before** the run loop so the queue holds only tasks that will do work. Each
reproduces a check `run_feature_comp_task.py` already makes internally — the point is to make it
before paying for a process launch and the somatic panel's feature load.

1. **Existing results** — the script's own skip test (three grid files plus held-out risk scores).
   This is where a running array's finished work gets subtracted. Read-only: uses
   `schemes.scheme_results_dir`, not `get_output_dir`, which would create directories.
2. **Event exists, and is powered enough** — the manifest is a static TSV listing events some
   schemes do not define, and `validate_cox_inputs` raises below `MIN_EVENTS_FOR_CV` positives or
   `MIN_NON_EVENTS_FOR_CV` censored.

An event that cannot be checked (an unreadable input, say) is **kept** in the queue and reported,
so a data problem surfaces as a real traceback in the run rather than being silently filtered out
here.

In [ ]:
from functools import lru_cache

from pipelines.training.slurm_array_utils import MIN_EVENTS_FOR_CV, MIN_NON_EVENTS_FOR_CV
from shared.polars_utils import finite_or_zero


# --- Gate 1: existing results ------------------------------------------------

def missing_outputs(scheme: str, event: str) -> list[str]:
    """Which of the four files run_feature_comp_task checks before skipping are absent."""
    comp_dir = os.path.join(schemes.scheme_results_dir(scheme, ANCHOR), "feature_comps", event)
    paths = [
        os.path.join(comp_dir, f"{MODALITY}_test.csv"),
        os.path.join(comp_dir, f"{MODALITY}_val.csv"),
        os.path.join(comp_dir, f"{MODALITY}_ipcw_reference.csv.gz"),
        os.path.join(schemes.feature_held_out_dir(scheme, event, ANCHOR),
                     f"{MODALITY}_risk_scores.csv"),
    ]
    return [os.path.basename(p) for p in paths if not os.path.exists(p)]


def census(task_list, header):
    remaining, partial = [], []
    for scheme, event in task_list:
        absent = missing_outputs(scheme, event)
        if not absent:
            continue
        remaining.append((scheme, event))
        # Partial: the script re-runs these, but *reuses* the grid when all three grid files
        # exist -- so one missing only risk scores re-runs cheaply.
        if len(absent) < 4:
            partial.append((scheme, event, absent))

    print(f"{header}: {len(task_list) - len(remaining)}/{len(task_list)} complete, "
          f"{len(remaining)} remaining")
    if partial:
        print(f"\n  {len(partial)} task(s) partially complete (will re-run):")
        for scheme, event, absent in partial[:15]:
            note = ("grid re-fit" if any(not f.endswith("_risk_scores.csv") for f in absent)
                    else "grid reused, risk scores only")
            print(f"    {scheme}:{event:<12} missing {', '.join(absent)}  [{note}]")
        if len(partial) > 15:
            print(f"    ... and {len(partial) - 15} more")
    return remaining


# --- The cohort the script actually fits on ----------------------------------

@lru_cache(maxsize=None)
def analysis_cohort_mrns() -> frozenset:
    """MRNs surviving load_feature_modalities_df's cohort restriction.

    Mirrors _get_common_feature_mrns (somatic & prs & stage & treatment) followed by the
    inner join against cancer_type_df.
    """
    def mrns(path: str) -> set:
        return set(pl.read_csv(path, columns=["DFCI_MRN"])["DFCI_MRN"])

    return frozenset(
        mrns(_feature_file("complete_somatic_data_df.csv.gz", ANCHOR))
        & mrns(os.path.join(config.FEATURE_PATH, "complete_germline_data_df.csv.gz"))
        & mrns(_feature_file("cancer_stage_df.csv.gz", ANCHOR))
        & mrns(_feature_file("categorical_treatment_data_by_line.csv.gz", ANCHOR))
        & mrns(_feature_file("cancer_type_df.csv.gz", ANCHOR))
    )


# --- Gate 2: event exists, and is powered enough -----------------------------

@lru_cache(maxsize=None)
def _scheme_parquet(scheme: str) -> str:
    return os.path.join(config.SURV_PATH, schemes.embedding_file(scheme, ANCHOR))


@lru_cache(maxsize=None)
def _scheme_columns(scheme: str) -> frozenset:
    return frozenset(pl.scan_parquet(_scheme_parquet(scheme)).collect_schema().names())


@lru_cache(maxsize=None)
def event_counts(scheme: str, event: str) -> tuple[int, int] | None:
    """(n_events, n_non_events) on the analysis cohort, or None if the event is absent."""
    tt_col = f"tt_{event}"
    available = _scheme_columns(scheme)
    if tt_col not in available or event not in available:
        return None

    wanted = ["DFCI_MRN", event, tt_col]
    needs_brain = event == "brainM" and "CANCER_TYPE_BRAIN" in available
    if needs_brain:
        wanted.append("CANCER_TYPE_BRAIN")

    mask = (
        pl.col(tt_col).cast(pl.Float64, strict=False).is_finite()
        & (pl.col(tt_col) > 0)
        & pl.col(event).cast(pl.Float64, strict=False).is_finite()
    )
    if needs_brain:
        mask = mask & (finite_or_zero("CANCER_TYPE_BRAIN").cast(pl.Boolean) == False)  # noqa: E712

    counts = (
        pl.scan_parquet(_scheme_parquet(scheme)).select(wanted)
        .filter(pl.col("DFCI_MRN").is_in(analysis_cohort_mrns()))
        .filter(mask)
        .select(pl.len().alias("n_rows"),
                pl.col(event).cast(pl.Float64, strict=False).sum().alias("n_events"))
        .collect()
    )
    n_rows = int(counts["n_rows"][0])
    n_events = int(counts["n_events"][0] or 0)
    return n_events, n_rows - n_events


def prevalence_filter(task_list):
    """Split tasks into (runnable, unrunnable) on event existence and CV prevalence."""
    keep, dropped, verdicts = [], [], {}

    for scheme, event in task_list:
        try:
            counts = event_counts(scheme, event)
        except Exception as exc:  # unreadable input -> keep the task, surface the error
            verdicts[(scheme, event)] = ("unchecked", f"{type(exc).__name__}: {exc}")
        else:
            if counts is None:
                verdicts[(scheme, event)] = ("absent", "not defined for this scheme")
            else:
                n_events, n_non_events = counts
                if n_events < MIN_EVENTS_FOR_CV:
                    verdicts[(scheme, event)] = ("underpowered",
                                                 f"{n_events} events < {MIN_EVENTS_FOR_CV}")
                elif n_non_events < MIN_NON_EVENTS_FOR_CV:
                    verdicts[(scheme, event)] = ("underpowered",
                                                 f"{n_non_events} censored < {MIN_NON_EVENTS_FOR_CV}")
                else:
                    verdicts[(scheme, event)] = ("ok", f"{n_events} events / {n_non_events} censored")
        status = verdicts[(scheme, event)][0]
        (dropped if status in ("absent", "underpowered") else keep).append((scheme, event))

    unchecked = {k: r for k, (st, r) in verdicts.items() if st == "unchecked"}
    if unchecked:
        print(f"{len(unchecked)} event(s) could not be checked — kept in the queue:")
        for (scheme, event), reason in sorted(unchecked.items()):
            print(f"  {scheme}:{event} — {reason}")

    for status, title in (("absent", "not defined for their scheme"),
                          ("underpowered", "below the CV minimums")):
        events = sorted(k for k, (st, _) in verdicts.items() if st == status)
        if not events:
            continue
        print(f"\n{len(events)} event(s) {title} — dropping {len(events)} task(s):")
        for scheme, event in events[:15]:
            print(f"  {scheme}:{event:<12} {verdicts[(scheme, event)][1]}")
        if len(events) > 15:
            print(f"  ... and {len(events) - 15} more")

    print(f"\nPrevalence: {len(keep)}/{len(task_list)} tasks runnable, {len(dropped)} dropped")
    return keep, dropped


not_done = census(tasks, "Existing results")
print(f"\nAnalysis cohort: {len(analysis_cohort_mrns()):,} patients")
pending, skipped_prevalence = prevalence_filter(not_done)

print(f"\nQueue: {len(pending)} task(s) ({len(tasks)} manifest "
      f"- {len(tasks) - len(not_done)} done - {len(skipped_prevalence)} unrunnable)")

## Smoke test one task

Runs a single task in the foreground with output streaming straight through, **not** captured.

Do this before the full queue. The array's `somatic` runs were failing uniformly across every
scheme/event, and the wrapper's `[error]` lines carry no diagnosis — the real traceback goes to the
subprocess's stdout. This cell puts that traceback on screen directly.

`SMOKE_OVERWRITE = True` forces the fit even if this pair is already done, so the cell always
exercises the code path rather than short-circuiting on the skip check.

In [ ]:
SMOKE_SCHEME = "death_met"
SMOKE_EVENT = "death"
SMOKE_OVERWRITE = True    # force the fit even if outputs exist
SMOKE_N_JOBS = N_JOBS

_smoke_args = ["--scheme", SMOKE_SCHEME, "--event", SMOKE_EVENT, "--modality", MODALITY,
               "--anchor", ANCHOR, "--n-jobs", str(SMOKE_N_JOBS),
               "--max-iter", str(MAX_ITER), "--backend", BACKEND]
if SMOKE_OVERWRITE:
    _smoke_args.append("--overwrite")

print(f"{SMOKE_SCHEME}:{SMOKE_EVENT}:{MODALITY}  (n_jobs={SMOKE_N_JOBS}, "
      f"overwrite={SMOKE_OVERWRITE})\n" + "=" * 72)

_smoke = run_module(MODULE, _smoke_args, capture=False)

print("=" * 72)
print(f"exit {_smoke['returncode']} in {_smoke['wall_s'] / 60:.1f} min")
if _smoke["returncode"] != 0:
    print("\nFailed — the traceback above is the real cause of the array's uniform failures.\n"
          "Do not run the full queue until it is resolved.")

## Run

One subprocess per pending task, `MAX_WORKERS` in flight. A failure on one does not kill the queue.

Each subprocess gets `N_JOBS` cores for its CV folds, so BLAS is pinned to `N_JOBS` threads rather
than 1 — this is the substantive difference from `01_feature_comparison.ipynb`, where every task is
single-core. `POLARS_MAX_THREADS`/`RAYON_NUM_THREADS` stay pinned low regardless: Polars sizes its
Rayon pool to the machine's core count at import, which with several tasks in flight is hundreds of
threads and trips "can't start new thread" before any fitting begins. The SLURM script does not pin
them because there one task owns the node.

`VERBOSE = False` collapses the run to a progress bar. Each subprocess's stdout is still captured —
it is the only record of why a task failed, and the summary cell below reads it.

In [ ]:
import concurrent.futures

from tqdm.auto import tqdm


def run_task(scheme: str, event: str) -> dict:
    args = ["--scheme", scheme, "--event", event, "--modality", MODALITY,
            "--anchor", ANCHOR, "--n-jobs", str(N_JOBS),
            "--max-iter", str(MAX_ITER), "--backend", BACKEND]
    if OVERWRITE:
        args.append("--overwrite")
    # BLAS gets N_JOBS threads (this modality is multi-core); Polars/Rayon stay pinned so
    # concurrent tasks do not each size a pool to the whole machine.
    thread_env = {var: str(N_JOBS) for var in (
        "OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS")}
    thread_env.update({"POLARS_MAX_THREADS": "1", "RAYON_NUM_THREADS": "1"})
    outcome = run_module(MODULE, args, env=thread_env, capture=True)
    outcome.update(scheme=scheme, event=event)
    return outcome


# OVERWRITE re-runs finished work by design, so it bypasses the existing-results gate.
# The prevalence gate still applies -- --overwrite does not make an underpowered event fittable.
_underpowered = set(skipped_prevalence)
queue_tasks = pending if not OVERWRITE else [t for t in tasks if t not in _underpowered]
n_workers = max(1, min(int(MAX_WORKERS), len(queue_tasks))) if queue_tasks else 1
print(f"Running {len(queue_tasks)} task(s) across {n_workers} worker(s) at {N_JOBS} core(s) each"
      + (" (OVERWRITE=True)" if OVERWRITE else ""))

run_started = time.perf_counter()
results, n_failed = [], 0

if queue_tasks:
    # Threads only wait on subprocess.run -- the work is in separate processes, so the GIL
    # is irrelevant and a thread pool avoids pickling overhead.
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_workers) as pool:
        in_flight = {pool.submit(run_task, *task): task for task in queue_tasks}
        with tqdm(total=len(queue_tasks), desc=MODALITY, unit="task") as bar:
            try:
                for future in concurrent.futures.as_completed(in_flight):
                    result = future.result()
                    results.append(result)
                    label = f"{result['scheme']}:{result['event']}"
                    if result["returncode"] != 0:
                        n_failed += 1
                        bar.write(f"[fail] {label} (exit {result['returncode']})")
                    if VERBOSE:
                        bar.write(f"\n{'=' * 72}\n{label} (exit {result['returncode']}, "
                                  f"{result['wall_s']:.1f}s)\n{'=' * 72}")
                        bar.write(result["stdout"].rstrip())
                    bar.update(1)
                    bar.set_postfix_str(f"{n_failed} failed" if n_failed else "", refresh=True)
            except KeyboardInterrupt:
                # Without this, pool shutdown would block on every queued task.
                for future in in_flight:
                    future.cancel()
                print("\nInterrupted -- cancelled queued tasks, waiting for in-flight ones.")
                raise

run_elapsed = time.perf_counter() - run_started
print(f"\nRan {len(results)} task(s) in {run_elapsed / 60:.1f} min across {n_workers} worker(s)")

## Summary

With `VERBOSE = False` this is where a failure is diagnosed — the run loop captured every
subprocess's output but printed none of it, so the tail of each failing task's stdout is reproduced
here. The full text stays in `results[i]["stdout"]`.

With both gates applied, a non-zero exit is more likely to be a real defect: already-done work and
underpowered events were removed before the loop started. What remains is post-imputation row loss
(the prevalence gate counts before modality NaN drops) and genuine failures;
`results/skipped_events/*.jsonl` records the reason either way.

If **every** task failed, suspect an input rather than any individual fit — the cohort intersection
in the preconditions cell is the usual culprit, and the smoke-test cell above is the fastest way to
see why.

The final census runs over the full task list, so "complete" includes whatever a running array
finished alongside this notebook.

In [ ]:
FAIL_TAIL_LINES = 15

failed = [r for r in results if r["returncode"] != 0]
print(f"{len(results) - len(failed)} succeeded, {len(failed)} failed")

if results and len(failed) == len(results):
    print("\n[!] Every task failed. That points at a shared input, not the fits themselves —\n"
          "    re-check the preconditions cell and run the smoke test for a live traceback.")

for result in failed:
    label = f"{result['scheme']}:{result['event']}"
    lines = result["stdout"].splitlines()
    tail = lines[-FAIL_TAIL_LINES:]
    print(f"\n{'-' * 72}\n{label} (exit {result['returncode']}) — "
          f"last {len(tail)} of {len(lines)} output line(s)\n{'-' * 72}")
    print("\n".join(tail) if tail else "(no output captured)")

if results:
    # Per-task wall time: with n_workers in flight these sum to more than the elapsed clock.
    total_task_min = sum(r["wall_s"] for r in results) / 60
    print(f"\nSlowest tasks (total task time {total_task_min:.1f} min across "
          f"{run_elapsed / 60:.1f} min wall):")
    for result in sorted(results, key=lambda r: r["wall_s"], reverse=True)[:5]:
        print(f"  {result['scheme']}:{result['event']:<12} {result['wall_s'] / 60:>6.1f} min")

# Coverage on disk now, over the full task list.
n_done = sum(1 for scheme, event in tasks if not missing_outputs(scheme, event))
pct = 100.0 * n_done / len(tasks) if tasks else 0.0
print(f"\nCoverage: {n_done}/{len(tasks)} {MODALITY} tasks complete ({pct:.1f}%)")